In [ ]:
""" 
from bs4 import BeautifulSoup
import json

# Membaca file HTML yang Anda berikan
with open("trends24.html", "r", encoding="utf-8") as f:
    html_content = f.read()

soup = BeautifulSoup(html_content, "html.parser")

# Mencari elemen tbody dengan class 'list'
tbody = soup.find("tbody", class_="list")

results = []

if tbody:
    # Mencari semua baris (tr) di dalam tbody
    rows = tbody.find_all("tr")
    
    for row in rows:
        # Ekstraksi data berdasarkan class kolom
        rank = row.find("td", class_="rank").text.strip()
        topic_element = row.find("td", class_="topic")
        topic_name = topic_element.text.strip()
        topic_link = topic_element.find("a")["href"] if topic_element.find("a") else None
        
        # Mengambil data tambahan (opsional)
        top_pos = row.find("td", class_="position").text.strip()
        tweet_count = row.find("td", class_="count").text.strip()
        duration = row.find("td", class_="duration").text.strip()

        # Simpan ke dictionary
        results.append({
            "rank": rank,
            "topic": topic_name,
            "link": topic_link,
            "top_position": top_pos,
            "tweet_count": tweet_count,
            "duration": duration
        })

# Cetak dalam format JSON yang rapi
print(json.dumps(results[:10], indent=4)) # Cetak 10 besar saja sebagai contoh
"""

[
    {
        "rank": "1",
        "topic": "#SFxPondPhuwinPermpoon",
        "link": "https://twitter.com/search?q=%23SFxPondPhuwinPermpoon",
        "top_position": "1",
        "tweet_count": "240",
        "duration": "21 hrs"
    },
    {
        "rank": "2",
        "topic": "#ShopeeBeli2Gratis1Jam8Malam",
        "link": "https://twitter.com/search?q=%23ShopeeBeli2Gratis1Jam8Malam",
        "top_position": "1",
        "tweet_count": "240",
        "duration": "24 hrs"
    },
    {
        "rank": "3",
        "topic": "#GrabFoodMegaGalex\u0e08\u0e2d\u0e2a\u0e01\u0e27\u0e34\u0e19",
        "link": "https://twitter.com/search?q=%23GrabFoodMegaGalex%E0%B8%88%E0%B8%AD%E0%B8%AA%E0%B8%81%E0%B8%A7%E0%B8%B4%E0%B8%99",
        "top_position": "1",
        "tweet_count": "240",
        "duration": "20 hrs"
    },
    {
        "rank": "4",
        "topic": "PPP FAMILY HOPPERS",
        "link": "https://twitter.com/search?q=PPP%20FAMILY%20HOPPERS",
        "top_position": "2",
        "

In [ ]:
"""
Cara scraping trends24
"""

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import json
import time

url = "https://trends24.in/indonesia/"

# Setup Chrome
chrome_options = Options()
chrome_options.add_argument("--headless") # Jalankan di latar belakang
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

try:
    print("Mengakses halaman...")
    driver.get(url)
    wait = WebDriverWait(driver, 20)

    # 1. Navigasi: Klik tab 'Table'
    tab_button = wait.until(EC.element_to_be_clickable((By.ID, "tab-link-table")))
    print("Mengklik tab Table...")
    driver.execute_script("arguments[0].click();", tab_button)

    # 2. Tunggu konten tabel dimuat
    print("Menunggu data dimuat...")
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "tbody.list tr")))
    time.sleep(2) # Jeda tambahan untuk memastikan semua row terisi

    # 3. Parsing Data
    soup = BeautifulSoup(driver.page_source, "html.parser")
    tbody = soup.find("tbody", class_="list")

    results = []
    if tbody:
        rows = tbody.find_all("tr")
        for row in rows:
            topic_cell = row.find("td", class_="topic")
            if not topic_cell: continue

            data = {
                "rank": row.find("td", class_="rank").get_text(strip=True),
                "topic": topic_cell.get_text(strip=True),
                "link": topic_cell.find("a")["href"] if topic_cell.find("a") else None,
                "top_position": row.find("td", class_="position").get_text(strip=True),
                "tweet_count": row.find("td", class_="count").get_text(strip=True),
                "duration": row.find("td", class_="duration").get_text(strip=True)
            }
            results.append(data)

        # 4. MENULIS KE FILE JSON
        nama_file = "trends24_indonesia.json"
        with open(nama_file, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=4, ensure_ascii=False)
        
        print(f"Berhasil! {len(results)} data telah disimpan ke '{nama_file}'.")
    else:
        print("Gagal mengambil data dari tabel.")

except Exception as e:
    print(f"Terjadi kesalahan: {e}")

finally:
    driver.quit()

Mengakses halaman...
Mengklik tab Table...
Menunggu data dimuat...
Berhasil! 138 data telah disimpan ke 'tren_indonesia.json'.


In [ ]:
"""
Cara membaca json hasil scraping trends24
"""
import json

# Membaca file JSON
with open("tren_indonesia.json", "r", encoding="utf-8") as f:
    data = json.load(f)
    
# Sekarang 'data' adalah list of dictionary
# Contoh: Mengambil 3 tren teratas
for item in data[:3]:
    print(f"Ranking {item['rank']}: {item['topic']}")
    print(f"Jumlah Tweet: {item['tweet_count']}")
    print("-" * 20)

Ranking 1: #SFxPondPhuwinPermpoon
Jumlah Tweet: 240
--------------------
Ranking 2: #GrabFoodMegaGalexจอสกวิน
Jumlah Tweet: 240
--------------------
Ranking 3: GF MEGA GALE X JOSSGAWIN
Jumlah Tweet: 240
--------------------
